# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

CRUSOE_API_KEY=****bmsK
LANGSMITH_API_KEY=****here
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain.agents import create_agent

In [3]:
from langchain_crusoe import ChatCrusoe
agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [4]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


*Steps up to the mic, adjusts it, and clears throat*

Good evening, everyone! I’m a full-stack comedian, which means I’m responsible for the setup, the punchline, and all the awkward silence in between. 

Here’s one for you:

Why did the full-stack developer break up with their partner?

The backend logic was solid, the chemistry was consistent, and they always maintained a secure connection. But they just couldn't figure out why their partner's frontend wasn't rendering affection properly. 

They spent weeks debugging the relationship, clearing their emotional cache, and trying to update their status from "It's Complicated" to "Committed." 

Turns out, they just needed to apply `display: flex` to their communication style, so they could finally align their expectations in the center. 

*Ba-dum-tss* 🥁

Thank you, thank you! I'll be here all week. Don't forget to tip your server... side developers! And make sure you leave a 200 OK on your way out!


## values
You have seen this streaming mode in our examples so far. 

In [5]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


================================== Ai Message ==================================

As a full-stack comedian, I handle both the front-end (the setup) and the back-end (the awkward silence after you groan). Here’s one for you:

Why did the programmer go broke?

Because he used up all his cache.


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [6]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


*As a full-stack comedian, I handle everything from the front

-end (the setup) to the back-end (the punchlines). Here’s a family

-friendly poem about a bear who has a bit of a scheduling conflict with his

 own bedtime.*

**The Bear Who

...

 juice on his fur and a smile on his face,
He waddled back home to

 his favorite place.

He dreamed about honey and buzzing of bees,
And slept through the winter

, a snore in the trees.

***
*Thank you, thank you!

 I'll be here all winter. Try the honey, and don't forget to tip your

 waitstaff!*

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [7]:
from langchain_crusoe import ChatCrusoe
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='64cb8192-6c9a-4e0b-a8f2-bc056f7dcd51')]})


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='64cb8192-6c9a-4e0b-a8f2-bc056f7dcd51'), AIMessage(content='Let me check the weather in San Francisco for you.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 161, 'total_tokens': 218, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 33, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'multimodal_tokens': None}}, 'model_provider': 'openai', 'model_name': 'zai/GLM-5.2', 'system_fingerprint': None, 'id': 'chatcmpl-___prefill_addr_10.234.11.50:8998___decode_addr_10.234.35.7:8998_37b150b4ca4c2631', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0395a-f2bc-7142-8dd4-bc1e60a364df-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Fran

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='64cb8192-6c9a-4e0b-a8f2-bc056f7dcd51'), AIMessage(content='Let me check the weather in San Francisco for you.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 161, 'total_tokens': 218, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 33, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'multimodal_tokens': None}}, 'model_provider': 'openai', 'model_name': 'zai/GLM-5.2', 'system_fingerprint': None, 'id': 'chatcmpl-___prefill_addr_10.234.11.50:8998___decode_addr_10.234.35.7:8998_37b150b4ca4c2631', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0395a-f2bc-7142-8dd4-bc1e60a364df-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Fran

In [8]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


('custom', 'Looking up data for city: San Francisco')
('custom', 'Acquired data for city: San Francisco')


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [9]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Looking up data for city: San Francisco
Acquired data for city: San Francisco


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
